# COMP8240 — Replication and extension of *Faking Fake News for Real Fake News Detection* (Huang et al., ACL 2023)

This notebook runs the complete pipeline used for the final report, in order.
Choose **Runtime → Change runtime type → T4 GPU**.
Every output is written to Google Drive, and every long step resumes if Colab disconnects.

* **Part A**: replication (train and evaluate RoBERTa-large on PropaNews).
* **Part B**: new data (LIAR, WELFake and the newly constructed WikiPropaNews), evaluation and analysis.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob
REPO_URL = 'https://github.com/Atharvach/COMP8240-FakingFakeNews-Assignment_Project'   # <-- this repository
DRIVE = '/content/drive/MyDrive/fakenewsproject'
OUT = f'{DRIVE}/new_data_out'
os.makedirs(OUT, exist_ok=True)

In [ ]:
# Clone this repo, then clone the original code at the pinned commit and apply the AdamW patch
%cd /content
!test -d COMP8240 || git clone -q $REPO_URL COMP8240
!bash COMP8240/replication/setup_and_patch.sh /content/FakingFakeNews
!cp -r COMP8240/new_data COMP8240/analysis /content/FakingFakeNews/
!pip install -q -U transformers accelerate bitsandbytes datasets nltk tabulate scipy

## Part A — Replication (about 75 min)
`--max_epoch 10`; every other hyperparameter is the repository default. The script prints Snopes results at the end.

In [ ]:
%cd /content/FakingFakeNews/roberta
!python train_roberta.py --output_dir $DRIVE/outputs --data_dir ../data/ --max_epoch 10 2>&1 | tee $DRIVE/train_log.txt

In [ ]:
# OPTION (skip training): download the checkpoint used in the report instead of running the cell above.
CKPT_URL = 'https://drive.google.com/file/d/1ktPquPDnze38ohhXelLIQbRh0IGp8ukk/view?usp=sharing'   # Google Drive share link to best.pt ('Anyone with the link')
os.makedirs(f'{DRIVE}/outputs/20261007_142926', exist_ok=True)
!gdown --fuzzy "$CKPT_URL" -O $DRIVE/outputs/20261007_142926/best.pt

In [ ]:
# Checkpoint used in the report: f'{DRIVE}/outputs/20261007_142926/best.pt'
CKPT = sorted(glob.glob(f'{DRIVE}/outputs/*/best.pt'))[-1]
print(CKPT, os.path.exists(CKPT))
# PolitiFact (test_roberta.py reads politifact_test.jsonl from --data_dir)
!python test_roberta.py --checkpoint_path "$CKPT" --data_dir ../data/

## Part B — New data
### Step 1: collect real articles from Wikinews (CPU, about 15 min)

In [ ]:
%cd /content/FakingFakeNews/new_data
!python 01_collect_wikinews.py --out $OUT/wikinews_raw.jsonl --start 2020-01-01 --contact your.email@students.mq.edu.au

### Step 2: generate PropaNews-style fakes with Qwen2.5-7B-Instruct (GPU, about 40 min)
Run the dry run first to inspect the prompts. The second command is the loaded-language top-up used in the report.

In [ ]:
!python 02_generate_fakes.py --raw $OUT/wikinews_raw.jsonl --out $OUT/generations.jsonl --pool_size 540 --dry_run

In [ ]:
!python 02_generate_fakes.py --raw $OUT/wikinews_raw.jsonl --out $OUT/generations.jsonl --pool_size 540

In [ ]:
!python 02_generate_fakes.py --raw $OUT/wikinews_raw.jsonl --out $OUT/generations.jsonl --pool_size 700 --force_type loaded-language

### Step 3: NLI filter and build WikiPropaNews; NLI pass rates per type (report Table 5)

In [ ]:
!python 03_build_dataset.py --gen $OUT/generations.jsonl --out_dir $OUT/wikipropanews --per_type 100
os.makedirs(f'{OUT}/results', exist_ok=True)
!python ../analysis/nli_by_type.py --gen $OUT/generations.jsonl --out_csv $OUT/results/nli_by_type.csv

**Restart the session** (Runtime → Restart session) to free GPU memory. Then re-run the first code cell and `%cd /content/FakingFakeNews/new_data`.

### Step 4: LLM-as-judge (Phi-3.5-mini-instruct, about 20 min)

In [ ]:
%cd /content/FakingFakeNews/new_data
!python 04_llm_judge.py --full $OUT/wikipropanews/wikipropanews_full.jsonl --out $OUT/judge.jsonl

### Step 5: human annotation and agreement
`make_sheet` writes a blind, shuffled sheet. Fill in the four answer columns yourself, save the result as `annotation_sheet_FILLED.csv` in the same folder, then score it.

In [ ]:
!python 05_annotation_agreement.py make_sheet --judge $OUT/judge.jsonl --out_dir $OUT/annotation

In [ ]:
!python 05_annotation_agreement.py score --judge $OUT/judge.jsonl --sheet $OUT/annotation/annotation_sheet_FILLED.csv --key $OUT/annotation/annotation_key.csv

### Existing datasets: LIAR (binary and strict) and WELFake (label meaning checked from the data)

In [ ]:
!python prep_existing.py --out_dir $OUT/existing

### Step 6: evaluate the replicated checkpoint on all six test sets (about 15 min)

In [ ]:
CKPT = sorted(glob.glob(f'{DRIVE}/outputs/*/best.pt'))[-1]
tests = {
 'snopes':        '../data/snopes_test.jsonl',
 'politifact':    '../data/politifact_test.jsonl',
 'liar':          f'{OUT}/existing/liar_test_binary.jsonl',
 'liar_strict':   f'{OUT}/existing/liar_test_strict.jsonl',
 'welfake':       f'{OUT}/existing/welfake_sample.jsonl',
 'wikipropanews': f'{OUT}/wikipropanews/wikipropanews_test.jsonl',
}
os.makedirs(f'{OUT}/preds', exist_ok=True)
for name, path in tests.items():
    print('=====', name)
    !python eval_any.py --checkpoint_path "$CKPT" --test_file "{path}" --out_csv "$OUT/preds/preds_{name}.csv"

### Step 7: tables, figures and the within-pair sign test

In [ ]:
!python 06_analyse.py --out_dir $OUT/results --full $OUT/wikipropanews/wikipropanews_full.jsonl --pred \
  Snopes=$OUT/preds/preds_snopes.csv PolitiFact=$OUT/preds/preds_politifact.csv LIAR=$OUT/preds/preds_liar.csv \
  LIAR-strict=$OUT/preds/preds_liar_strict.csv WELFake=$OUT/preds/preds_welfake.csv WikiPropaNews=$OUT/preds/preds_wikipropanews.csv
!python ../analysis/paired_sign_test.py --pred $OUT/preds/preds_wikipropanews.csv --full $OUT/wikipropanews/wikipropanews_full.jsonl --out_csv $OUT/results/paired_sign_test.csv

In [ ]:
import sys, torch, transformers, sklearn
print('Python', sys.version.split()[0], '| torch', torch.__version__, '| transformers', transformers.__version__, '| sklearn', sklearn.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')